In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 40. Week 28 — Conditional volatility, breaks, and regime-dependent evaluation

## 学習目標

- GARCH(1,1)のvariance recursionとstationarity条件を説明できる
- daily squared change proxyとintraday realized volatilityを区別できる
- methodology break前後のdiagnosticを分けられる
- conditional variance forecastをpoint forecastと別metricで評価できる

## 前提知識

- maximum likelihood、conditional expectation
- Treasury methodology break contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 40


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. GARCH contract

$$
h_t=\omega+\alpha\varepsilon_{t-1}^2+\beta h_{t-1},\qquad
\omega>0,\ \alpha,\beta\ge0,\ \alpha+\beta<1.
$$

本データは公表日ごとのyieldだけで、intraday returnを持たない。したがって ((\Delta y_t)^2) は日次変化のnoisy proxyであり、realized volatilityとは呼ばない。

In [4]:
ten_year_change_bp = curve_changes_bp[:, 3]
training_changes = ten_year_change_bp[change_dates <= train_end_date]
garch = qt.fit_garch11(training_changes)
display(
    pd.DataFrame(
        [
            {
                "omega": garch.omega,
                "alpha": garch.alpha,
                "beta": garch.beta,
                "persistence": garch.alpha + garch.beta,
                "converged": garch.converged,
                "iterations": garch.n_iterations,
            }
        ]
    )
)
assert garch.alpha + garch.beta < 1.0

,omega,alpha,beta,persistence,converged,iterations
0,1.237278,0.083505,0.851631,0.935136,True,19


In [5]:
audit_mask = change_dates <= validation_end_date
audit_changes = ten_year_change_bp[audit_mask]
conditional_variance = np.empty(audit_changes.size)
conditional_variance[0] = np.var(training_changes, ddof=1)
for index in range(1, audit_changes.size):
    conditional_variance[index] = (
        garch.omega
        + garch.alpha * audit_changes[index - 1] ** 2
        + garch.beta * conditional_variance[index - 1]
    )
rolling_proxy = pd.Series(audit_changes).rolling(20).std().to_numpy()

fig = go.Figure()
fig.add_scatter(x=change_dates[audit_mask], y=np.sqrt(conditional_variance), name="GARCH conditional sigma", mode="lines")
fig.add_scatter(x=change_dates[audit_mask], y=rolling_proxy, name="20-publication rolling sigma", mode="lines")
fig.add_vline(x=qt.TREASURY_METHOD_BREAK.timestamp() * 1000, line_dash="dash", line_color="black")
fig.update_layout(
    title="10y change volatility diagnostics through validation",
    xaxis_title="Treasury publication date",
    yaxis_title="Volatility proxy (bp)",
    template="plotly_white",
)
fig.show()

## 2. Break-aware audit

In [6]:
break_date = qt.TREASURY_METHOD_BREAK.to_datetime64()
period_rows = []
for period, mask in [
    ("pre-methodology-break", (change_dates <= train_end_date) & (change_dates < break_date)),
    ("post-methodology-break validation", (change_dates > train_end_date) & (change_dates <= validation_end_date) & (change_dates >= break_date)),
]:
    values = ten_year_change_bp[mask]
    period_rows.append(
        {
            "period": period,
            "observations": values.size,
            "mean_bp": values.mean(),
            "standard_deviation_bp": values.std(ddof=1),
            "mean_squared_change": np.mean(values**2),
        }
    )
display(pd.DataFrame(period_rows))

,period,observations,mean_bp,standard_deviation_bp,mean_squared_change
0,pre-methodology-break,1654,-0.046554,4.488435,20.136034
1,post-methodology-break validation,469,0.773987,7.482179,56.462687


## 3. 失敗モード

- squared daily yield changeをrealized volatilityと呼ぶ
- (alpha+\beta\ge1) のfitを無条件に長期varianceへ外挿する
- method change前後を同質と仮定する
- volatility forecastをdirection/level forecastと混ぜる
- stress periodを見てregime thresholdを後付けする

## 4. 段階別演習

### 基礎

1. GARCHのunconditional varianceを導出せよ。
2. rolling standard deviationとconditional sigmaを比較せよ。

### 標準

3. Gaussian QLIKEを定義しvalidationでconstant varianceと比較せよ。
4. methodology break前後でparameter stabilityを測れ。

### 研究

5. intraday dataを得た場合のmicrostructure-noise robust estimatorを調査せよ。

## 5. Exit Criteria

- [ ] GARCH parameter constraintを検査した
- [ ] volatility proxyの観測限界を明記した
- [ ] methodology breakを可視化した
- [ ] varianceとmean forecastの評価を分けた
- [ ] regimeを観測真値と呼んでいない

## 6. 出典

- [Engle (1982), ARCH](https://doi.org/10.2307/1912773)
- [Bollerslev (1986), Generalized ARCH](https://doi.org/10.1016/0304-4076(86)90063-1)

- [Forecasting: Principles and Practice — Stationarity and differencing](https://otexts.com/fpp3/stationarity.html)
- [Forecasting: Principles and Practice — ARIMA models](https://otexts.com/fpp3/arima.html)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)